In [13]:
import pandas as pd

# 1. Carga de Bases de Datos Normalizadas:
# Importamos la información ya procesada en la etapa de limpieza.
df_mayor = pd.read_csv("../data/raw/libro_mayor.csv")
df_banco = pd.read_csv("../data/raw/extracto_bancario.csv")

# Aseguramos el formato Datetime para permitir cálculos de series temporales
df_mayor['fecha'] = pd.to_datetime(df_mayor['fecha'])
df_banco['fecha'] = pd.to_datetime(df_banco['fecha'])

# Principio del Espejo Contable aplicado al motor:
# Igualamos el criterio de signo para comparar Mayor vs Banco en una misma convención.
df_banco['monto'] = df_banco['monto'] * -1

# --- NIVEL 1: MATCH EXACTO (CONTROL DE INTEGRIDAD) ---
# Ejecutamos un cruce de datos por Fecha y Monto. 
# Este primer filtro identifica todas las transacciones que no presentan 
# ninguna fricción ni demora administrativa.
match_exacto = pd.merge(
    df_mayor.reset_index(), 
    df_banco.reset_index(), 
    on=['fecha', 'monto'], 
    how='inner', 
    suffixes=('_mayor', '_banco')
)

print(f"✅ Nivel 1 terminado. Movimientos conciliados: {len(match_exacto)}")

✅ Nivel 1 terminado. Movimientos conciliados: 263


In [14]:
# --- NIVEL 2: EL MOTOR DE CERCANÍA (TRATAMIENTO DE CLEARING BANCARIO) ---

# 1. Aislamiento de Excepciones:
# Filtramos los registros que no lograron coincidir en el Nivel 1. 
# Estos representan el "ruido" que un administrativo normalmente revisaría a mano.
df_mayor_no_match = df_mayor.loc[~df_mayor.index.isin(match_exacto['index_mayor'])].copy()
df_banco_no_match = df_banco.loc[~df_banco.index.isin(match_exacto['index_banco'])].copy()

# 2. Requisito de Orden Cronológico:
# Para aplicar lógica de "AsOf" (Búsqueda hacia adelante), es imperativo 
# que las bases de datos estén ordenadas por fecha.
df_mayor_no_match = df_mayor_no_match.sort_values(['fecha', 'monto']).reset_index(drop=True)
df_banco_no_match = df_banco_no_match.sort_values(['fecha', 'monto']).reset_index(drop=True)

# 3. Ejecución del Algoritmo de Cercanía:
# Simulamos el clearing bancario: Buscamos montos idénticos en el banco que 
# hayan impactado hasta 3 días después de su registro en el Libro Mayor.
match_asof_3d = pd.merge_asof(
    df_mayor_no_match,
    df_banco_no_match,
    on="fecha",
    by="monto",
    tolerance=pd.Timedelta(days=3),
    direction="forward",
    suffixes=("_mayor", "_banco")
)

# 4. Consolidación de Resultados:
# Identificamos los movimientos que fueron rescatados gracias a la tolerancia temporal.
conciliados_nivel_2 = match_asof_3d.dropna(subset=['descripcion_banco'])

print(f"✅ Nivel 2 terminado.")
print(f"Movimientos rescatados por clearing: {len(conciliados_nivel_2)}")
match_asof_3d

✅ Nivel 2 terminado.
Movimientos rescatados por clearing: 736


,fecha,monto,descripcion_mayor,descripcion_banco
0,2026-03-01,-74068.84,Pago - Proveedor_5,BCO_MOV_PROV_REF680
1,2026-03-01,-64158.12,Pago - Aguas Cordobesas,BCO_MOV_AGUA_REF97
2,2026-03-01,-54937.94,Pago - Ecogas,BCO_MOV_ECOG_REF970
3,2026-03-01,-53693.50,Pago - Proveedor_7,BCO_MOV_PROV_REF958
4,2026-03-01,-48309.97,Pago - Proveedor_4,BCO_MOV_PROV_REF957
...,...,...,...,...
732,2026-03-25,60652.76,Cobro - Cliente_48,BCO_MOV_CLIE_REF226
733,2026-03-25,64998.56,Cobro - Cliente_40,BCO_MOV_CLIE_REF864
734,2026-03-25,93013.24,Cobro - Cliente_24,BCO_MOV_CLIE_REF833
735,2026-03-25,96991.16,Cobro - Cliente_49,BCO_MOV_CLIE_REF1000


In [15]:
import os

# --- FASE FINAL: EXPORTACIÓN DE EVIDENCIA Y PARTIDAS PENDIENTES ---

# 1. Identificación de Partidas Pendientes (Review Humano):
# Aislamos las filas que no encontraron pareja en ningún nivel. 
# Estas representan errores de tipeo, comisiones bancarias u omisiones de registro.
pendientes_mayor = match_asof_3d[match_asof_3d['descripcion_banco'].isna()].copy()
# Nos quedamos solo con las columnas del Mayor para que el Excel quede limpio
pendientes_mayor = pendientes_mayor[['fecha', 'monto', 'descripcion_mayor']]

banco_usados_nivel2 = conciliados_nivel_2['descripcion_banco'].tolist()
pendientes_banco = df_banco_no_match[
    ~df_banco_no_match['descripcion'].isin(banco_usados_nivel2)
].copy()

print("-" * 50)
print("📊 RESUMEN DE EXCEPCIONES PARA AUDITORÍA MANUAL:")
print(f"⚠️ En Libro Mayor (Posibles errores o no procesados): {len(pendientes_mayor)}")
print(f"⚠️ En Banco (Comisiones, impuestos o cobros no registrados): {len(pendientes_banco)}")
print("-" * 50)

# 2. Exportación a Formato Estándar:
# Generamos un reporte en Excel con pestañas separadas para que el área 
# contable pueda realizar los asientos de ajuste necesarios.
ruta_carpeta = "../data/processed"
os.makedirs(ruta_carpeta, exist_ok=True)
ruta_archivo = f"{ruta_carpeta}/Partidas_Pendientes_Marzo.xlsx"
ruta_conciliados = f"{ruta_carpeta}/conciliados_final.csv"

# 4. Exportamos a Excel + CSV de conciliados para Dashboard
try:
    with pd.ExcelWriter(ruta_archivo, engine='openpyxl') as writer:
        pendientes_mayor.to_excel(writer, sheet_name='Revisar_en_Mayor', index=False)
        pendientes_banco.to_excel(writer, sheet_name='Revisar_en_Banco', index=False)

    # Consolidamos la "Evidencia de Éxito" para alimentar el Dashboard visual.
    cols_conciliacion = ['fecha', 'monto', 'descripcion_mayor', 'descripcion_banco']
    conciliados_n1 = match_exacto[cols_conciliacion].copy()
    conciliados_n2 = conciliados_nivel_2[cols_conciliacion].copy()
    conciliados_final = pd.concat([conciliados_n1, conciliados_n2], ignore_index=True)
    conciliados_final = conciliados_final.drop_duplicates()
    conciliados_final.to_csv(ruta_conciliados, index=False)

    print(f"✅ Reporte de Auditoría generado: {ruta_archivo}")
    print(f"✅ Dataset de Conciliados guardado para Dashboard: {ruta_conciliados}")
except Exception as e:
    print(f"❌ Error al exportar: {e}")

--------------------------------------------------
📊 RESUMEN DE EXCEPCIONES PARA AUDITORÍA MANUAL:
⚠️ En Libro Mayor (Posibles errores o no procesados): 1
⚠️ En Banco (Comisiones, impuestos o cobros no registrados): 2
--------------------------------------------------
✅ Reporte de Auditoría generado: ../data/processed/Partidas_Pendientes_Marzo.xlsx
✅ Dataset de Conciliados guardado para Dashboard: ../data/processed/conciliados_final.csv
